# 🐍 Mamba: Selective State Space Models
## Versión alumno · Aprende cómo funciona la alternativa a Transformers

En este notebook vas a:

1. 📖 **Entender** qué son los State Space Models (SSM)
2. 🔍 **Descubrir** cómo Mamba selecciona qué recordar
3. ⚡ **Comparar** la eficiencia vs Transformers
4. 🚪 **Explorar** cómo funcionan las puertas de selección
5. ✏️ **Experimentar** con los tres modelos

---

### 💡 La gran pregunta

Los Transformers son potentes pero... **¿necesitan ser tan caros?**

Mamba propone una alternativa que:
- Procesa secuencias en **tiempo lineal** O(n), no O(n²)
- Mantiene la capacidad de recordar información importante
- Usa **selección de contenido** para decidir qué guardar

> 💬 Imagina que tu cerebro pudiera decidir automáticamente qué recuerdos son importantes...
> ¡Eso es lo que hace Mamba!

## 1. Preparación

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import math

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"📱 Dispositivo: {device}")

📱 Dispositivo: cpu


## 2. El dataset: Don Quijote

Usamos **El Quijote** con tokenización a nivel de carácter.

Ventajas:
- Vocabulario pequeño (~100 caracteres)
- Tarea sencilla: predecir siguiente carácter
- El Quijote está en español, ¡lo conoces!

In [12]:
# ======================================
# 2.1 Descargar El Quijote
# ======================================

data_path = Path("datos/el_quijote.txt")

if not data_path.exists():
    print("📥 Descargando Don Quijote...")
    url = "https://www.gutenberg.org/files/2000/2000-0.txt"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text = response.text
    
    start_marker = "*** START OF THE PROJECT GUTENBERG"
    end_marker = "*** END OF THE PROJECT GUTENBERG"
    start_idx = text.find(start_marker) + len(start_marker)
    end_idx = text.find(end_marker)
    text = text[start_idx:end_idx].strip()
    data_path.write_text(text)
    print("✅ Descarga completada")

text = data_path.read_text()
print(f"📚 El Quijote: {len(text):,} caracteres")

# ======================================
# 2.2 Vocabulario y codificación
# ======================================
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)
print(f"🔤 Vocabulario: {vocab_size} caracteres únicos")

# ======================================
# 2.3 Función para obtener batches
# ======================================
def get_batch(batch_size=64, block_size=128, device='cpu'):
    """
    Obtiene fragmentos aleatorios del texto.
    
    - x: los primeros block_size caracteres
    - y: los mismos caracteres pero desplazados 1 posición (el objetivo)
    """
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# Ejemplo rápido
x_ex, y_ex = get_batch(batch_size=2, block_size=20, device=device)
print(f"\n🔍 Ejemplo:")
print(f"   x: {''.join(itos[i.item()] for i in x_ex[0][:15])}")
print(f"   y: {''.join(itos[i.item()] for i in y_ex[0][:15])}")

📚 El Quijote: 1,038,397 caracteres
🔤 Vocabulario: 89 caracteres únicos

🔍 Ejemplo:
   x: das las razones
   y: as las razones 


---

## 3. State Space Models: la base teórica

Antes de Mamba, necesitamos entender qué son los **State Space Models**.

### 3.1 La idea fundamental

Un SSM es como un **sistema con memoria interna**: tiene un **estado** que se actualiza con cada nuevo input.

Piensa en tu memoria de trabajo:
```
1. Ves algo nuevo → tu cerebro lo procesa
2. Decide qué guardar en tu "estado" mental
3. Cada cosa nueva modifica ese estado
4. Para responder, usas el estado actual
```

### 3.2 Las ecuaciones (versión continua)

Matemáticamente, un SSM continuo funciona así:

```
h'(t) = A·h(t) + B·u(t)    ← Cómo cambia el estado
y(t)    = C·h(t) + D·u(t)   ← Cómo sale el resultado
```

Donde:
- **h(t)**: el estado interno (la memoria)
- **u(t)**: la entrada actual
- **y(t)**: la salida
- **A, B, C, D**: parámetros que aprendemos

### 3.3 Versión discreta (la que usamos)

Cuando procesamos secuencias discretas (texto, audio):

```
h_t = A_t·h_{t-1} + B_t·u_t     ← Nuevo estado
y_t  = C_t·h_t + D_t·u_t        ← Nueva salida
```

El subíndice _t indica que dependemos del **tiempo/paso actual**.

### 🤔 Ilustración: Cómo funciona un SSM

```
Secuencia de entrada: "En un lu-gar de la Mancha..."

Paso 1: Procesar 'E'
   Estado: [Info sobre 'E']
   
Paso 2: Procesar 'n' 
   Estado: [Info sobre 'E', 'n']
   
Paso 3: Procesar '-' 
   Estado: [Info sobre 'E', 'n', '-']
   
Paso 4: Procesar 'g' 
   Estado: [Info relevante para predecir siguiente]
   
   El SSM ha "comprimido" toda la información
   previa en un estado de tamaño fijo.
```

**Clave**: Aunque la secuencia sea infinita, el estado tiene **tamaño fijo**. El SSM decide qué información comprimir y cómo.

---

## 4. La innovación de Mamba

Aquí está la diferencia crucial con los SSM tradicionales.

### 4.1 SSM tradicionales vs Mamba

| SSM Tradicional | Mamba |
|----------------|-------|
| A, B, C, D son **fijos** | A, B, C son **input-dependent** |
| Misma "memoria" para todo | Puede "olvidar" o "recordar" selectivamente |
| Más rápido pero menos expresivo | Más expresivo con misma complejidad |

### 4.2 ¿Qué significa "input-dependent"?

En Mamba:
```
B_t = Red neuronal(x_t)   ← Depende del input
C_t = Red neuronal(x_t)  ← Depende del input
A_t = parametrizado(x_t) ← También depende
dt_t = softplus(x_t)     ← Tiempo de retención
```

Esto significa que **el modelo decide dinámicamente**:
- ¿Cuánto del input actual debo guardar?
- ¿Cuánto del estado anterior mantener?
- ¿Cuánto debo "olvidar"?

### 4.3 El parámetro dt (delta)

**dt** es como el "tiempo de retención" de cada input:

- dt grande → el input influye más en el estado
- dt pequeño → el input se "olvida" más rápido

Con softplus, dt siempre es positivo y permite valores muy variados.

### 🔬 Analogía: El archivador selectivo

Imagina que tienes un archivador con **16 cajones** (d_state=16):

**SSM tradicional:**
- Guardas TODO en los cajones
- Cuando se llena, sobrescribes
- No eliges qué es importante

**Mamba:**
- Para cada documento nuevo, decides:
  - ¿Cuáles cajones son relevantes? (B_t)
  - ¿Qué extraer del documento? (C_t)
  - ¿Cuánto tiempo guardarlo? (dt_t)
- Puedes "recordar" cosas importantes más tiempo
- Puedes "olvidar" cosas irrelevantes rápido

---

## 5. El bloque Mamba (código)

Ahora veamos cómo se implementa. No te preocupes si no entiendes todo,
lo importante es la **intuición**.

In [13]:
# ======================================
# 5.1 Normalización simple (RMSNorm)
# ======================================

class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization
    
    Versión simplificada de LayerNorm: solo usa la media de cuadrados.
    """
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
    
    def forward(self, x):
        # Normalizar por la raíz del error cuadrático medio
        output = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * output

In [14]:
# ======================================
# 5.2 El bloque Mamba
# ======================================

class MambaBlock(nn.Module):
    """
    Bloque Mamba: la pieza fundamental de los Selective SSM
    
    Pipeline:
    1. Proyección de entrada: x → (x_inner, z)
       - x_inner: para calcular B, C, dt
       - z: para la puerta de salida
    
    2. Convolución local: preprocessar x_inner
       - Captura patrones locales cortos
    
    3. Selección SSM:
       - Proyectar x_inner → B, C, dt
       - dt = softplus para tiempo de retención
       - A parametrizado (matriz de estado)
       - SSM scan: actualizar estado con selección
    
    4. Puerta de salida:
       - y = SSM_output * sigmoid(z)
       - + input * sigmoid(D) (skip connection)
    """
    
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state      # Tamaño del estado
        self.d_inner = int(expand * d_model)
        self.d_conv = d_conv        # Tamaño del kernel de conv
        
        # ===== PROYECCIÓN DE ENTRADA =====
        # Duplicamos: mitad para proceso SSM, mitad para gate
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        
        # ===== CONVOLUCIÓN LOCAL =====
        # Depthwise conv: cada canal procesa independientemente
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=d_conv,
            padding=d_conv - 1,
            groups=self.d_inner,
        )
        
        # ===== PROYECCIÓN PARA PARÁMETROS SSM =====
        # Salida: d_state (B) + d_state (C) + 1 (dt) = 2*d_state + 1
        self.x_proj = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)
        
        # ===== MATRIZ DE ESTADO A =====
        # Inicializada con valores negativos (decaimiento)
        self.A_log = nn.Parameter(torch.randn(self.d_inner, d_state))
        # D es para la skip connection final en espacio d_model
        self.D = nn.Parameter(torch.ones(self.d_model))
        
        # ===== PROYECCIÓN DE SALIDA =====
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
    
    def forward(self, x):
        b, l, d = x.shape
        
        # ===== PASO 1: SPLIT =====
        xz = self.in_proj(x)
        x_inner, z = xz.chunk(2, dim=-1)
        
        # ===== PASO 2: CONVOLUCIÓN LOCAL =====
        x_inner = x_inner.transpose(1, 2)  # (b, l, d) → (b, d, l)
        x_inner = self.conv1d(x_inner)[:, :, :l]  # Truncar si es necesario
        x_inner = x_inner.transpose(1, 2)  # Volver a (b, l, d)
        x_inner = F.silu(x_inner)  # Activación Swish
        
        # ===== PASO 3: SELECCIÓN SSM =====
        # Proyectar para obtener B, C, dt
        x_decay = self.x_proj(x_inner)
        B, C, dt = x_decay.split([self.d_state, self.d_state, 1], dim=-1)
        dt = F.softplus(dt)  # Siempre positivo
        
        # Matriz A (inicializada como negativa para decaimiento)
        A = -torch.exp(self.A_log.float())
        
        # Ejecutar SSM con selección
        y = self.ssm_select(x_inner, dt, A, B, C)
        
        # ===== PASO 4: PUERTA Y SALIDA =====
        # Gate: decidir qué pasa directamente (z)
        y = y * torch.sigmoid(z)
        y = self.out_proj(y)
        # Skip connection final en el espacio del modelo (d_model)
        y = y + x * torch.sigmoid(self.D)
        
        return y
    
    def ssm_select(self, x, dt, A, B, C):
        """
        SSM con selección de contenido.
        
        Para cada posición t:
        1. dt[t] = cuánto "tiempo" dedicamos a este input
        2. B[t] = qué del input va al estado
        3. C[t] = qué del estado sale como output
        4. A = matriz de transición (cómo fluye el estado)
        
        h_t = dt[t] * (A * h_{t-1} + B[t] * x[t])
        y_t = C[t] * h_t
        """
        b, l, d_inner = x.shape
        d_state = self.d_state
        
        # Inicializar estado a cero
        h = torch.zeros(b, d_inner, d_state, device=x.device, dtype=x.dtype)
        outputs = []
        
        for i in range(l):
            dti = dt[:, i, :]        # Tiempo de retención
            Bi = B[:, i, :]         # Input → estado
            Ci = C[:, i, :]         # Estado → output
            
            # ===== ECUACIÓN DE ESTADO =====
            # h_new = dt * (A @ h_prev + B * x)
            h_new = dti.unsqueeze(-1) * (
                A.unsqueeze(0) * h + 
                (Bi.unsqueeze(1) * x[:, i, :].unsqueeze(-1))
            )
            
            # ===== ECUACIÓN DE SALIDA =====
            # y = C @ h
            y_i = (Ci.unsqueeze(1) * h_new).sum(-1)
            outputs.append(y_i)
            
            h = h_new
        
        return torch.stack(outputs, dim=1)

### 📊 ¿Por qué Mamba es más rápido?

**Atención global (Transformers):**
```
Cada token compara con TODOS los demás
Para 128 tokens: 128 × 128 = 16,384 operaciones
```

**Mamba (SSM con selección):**
```
El estado tiene tamaño FIJO (ej: 16)
Para 128 tokens: 128 × 16 = 2,048 operaciones

→ ¡8 veces menos operaciones!
```

Y a medida que la secuencia crece, la ventaja es mayor:

| Longitud | Atención O(n²) | Mamba O(n) | Speedup |
|----------|---------------|------------|--------|
| 128 | 16K | 2K | 8x |
| 512 | 262K | 8K | 32x |
| 2048 | 4M | 33K | 128x |

---

## 6. El bloque de Atención (para comparar)

In [ ]:
# ======================================
# 6. BLOQUE DE ATENCIÓN (Referencia Transformer)
# ======================================

class AttentionBlock(nn.Module):
    """
    Bloque de atención multi-cabeza clásico.
    
    Cada token puede "mirar" a todos los demás tokens.
    Esto es la base de los Transformers modernos.
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        # Atención multi-cabeza
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)        
        
        # Normalización
        self.norm = nn.LayerNorm(d_model)
        
        # Feed-forward
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
    
    def forward(self, x):
        # Auto-atención
        attn_out, _ = self.attn(x, x, x)
        x = self.norm(x + attn_out)
        x = self.norm(x + self.ff(x))
        return x

SyntaxError: invalid character '↓' (U+2193) (2358223339.py, line 15)

---

## 7. Los tres modelos

Ahora construimos los modelos completos para comparar.

In [17]:
# ======================================
# 7. MODELOS COMPLETOS
# ======================================

class MiniTransformer(nn.Module):
    """
    Transformer pequeño: solo atención.
    
    Estructura:
    Embedding → [AttentionBlock × 6] → Norm → Head
    
    Referencia de CALIDAD (pero más lento).
    """
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=6):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, 512, d_model) * 0.02)
        self.layers = nn.ModuleList([
            AttentionBlock(d_model, n_heads) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        b, l = x.shape
        x = self.embed(x) + self.pos_embed[:, :l, :]
        for layer in self.layers:
            x = layer(x)
        return self.head(self.norm(x))


class MiniMamba(nn.Module):
    """
    Mamba puro: solo bloques SSM selectivos.
    
    Estructura:
    Embedding → [MambaBlock × 12] → Norm → Head
    
    Referencia de EFICIENCIA (pero menos expresivo).
    """
    def __init__(self, vocab_size, d_model=128, n_layers=12, **mamba_kwargs):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            MambaBlock(d_model, **mamba_kwargs) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            x = layer(x)
        return self.head(self.norm(x))


class HybridMamba(nn.Module):
    """
    Híbrido: Mamba la mayor parte, atención estratégica.
    
    Estructura (12 capas):
    Capa 1: Attention
    Capa 2: Mamba
    Capa 3: Mamba
    Capa 4: Attention
    ...
    
    Equilibrio entre eficiencia y expresividad.
    """
    def __init__(self, vocab_size, d_model=128, n_layers=12, **mamba_kwargs):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        
        layers = []
        for i in range(n_layers):
            if i % 3 == 0:
                layers.append(AttentionBlock(d_model, n_heads=4))
            else:
                layers.append(MambaBlock(d_model, **mamba_kwargs))
        
        self.layers = nn.ModuleList(layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            x = layer(x)
        return self.head(self.norm(x))

### 📊 Resumen comparativo

| Modelo | Tipo | Ventaja | Limitación |
|--------|------|---------|------------|
| **MiniTransformer** | Solo atención | Captura cualquier dependencia | O(n²) - Caro |
| **MiniMamba** | Solo SSM | O(n) - Muy eficiente | Puede "olvidar" dependecias |
| **HybridMamba** | Mixto | Equilibrio | Más complejo |

---

## 8. Entrenamiento y generación

In [18]:
# ======================================
# 8. FUNCIÓN DE ENTRENAMIENTO
# ======================================

def train_and_generate(model, name, epochs=8, batch_size=64, block_size=128, 
                       lr=4e-4, device='cuda', temperature=0.8, save_dir="models_mamba"):
    """
    Entrena el modelo y genera texto al final.
    
    Args:
        model: modelo a entrenar
        name: nombre para mostrar
        epochs: pasadas por el dataset
        lr: learning rate
        temperature: 控制 aleatoriedad (0.8 = equilibrado)
    """
    import os
    os.makedirs(save_dir, exist_ok=True)
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    print(f"\n{'='*50}")
    print(f"🚀 ENTRENANDO: {name}")
    print(f"{'='*50}")
    
    losses = []
    best_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        steps = 80  # Pasos por época
        
        for _ in tqdm(range(steps), desc=f"  Época {epoch+1}/{epochs}"):
            x, y = get_batch(batch_size, block_size, device)
            
            # Forward
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
        
        scheduler.step()
        avg_loss = total_loss / steps
        losses.append(avg_loss)
        
        print(f"  Época {epoch+1:2d} | Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), f"{save_dir}/{name.replace(' ', '_')}_best.pt")
            print(f"    ⭐ Mejor modelo (loss: {best_loss:.4f})")
    
    # Gráfica
    plt.figure(figsize=(10, 4))
    plt.plot(range(1, epochs+1), losses, marker='o', linewidth=2)
    plt.title(f"📉 Curva de pérdida - {name}")
    plt.xlabel("Época")
    plt.ylabel("Cross-Entropy Loss")
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Generación
    print(f"\n📝 Generando texto (temperatura={temperature})")
    print("-" * 60)
    model.eval()
    with torch.no_grad():
        context = torch.tensor([stoi[' ']], dtype=torch.long, device=device).unsqueeze(0)
        generated = []
        
        for _ in range(400):
            logits = model(context)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            context = torch.cat([context, next_idx], dim=1)
            generated.append(itos[next_idx.item()])
        
        print(''.join(generated))
        print("\n" + "-" * 60)

### 🤔 ¿Qué es la temperatura?

Controla la **aleatoriedad** de la generación:

| Valor | Efecto |
|-------|--------|
| **0.5** | Muy seguro, repetitivo |
| **0.8** | Equilibrado ✓ |
| **1.2** | Creativo, puede ser incoherente |

---

## 9. 🏃 EJERCICIO FINAL: Entrena y compara

Ejecuta la celda siguiente para entrenar los tres modelos.

Después de ver los resultados, responde:
1. ¿Cuál converge más rápido?
2. ¿Cuál genera texto más coherente?
3. ¿Mamba puede "competir" con Transformer aquí?

In [19]:
# ======================================
# 9. ENTRENAR LOS TRES MODELOS
# ======================================

if __name__ == "__main__":
    d_model = 128
    save_dir = "saved_models_mamba"
    EPOCHS = 8
    TEMPERATURE = 0.85
    
    print(f"📱 Dispositivo: {device}")
    print(f"🔤 Vocabulario: {vocab_size} caracteres")
    
    models = [
       # ("MiniTransformer", MiniTransformer(vocab_size, d_model, n_heads=4, n_layers=6)),
        ("Pure Mamba", MiniMamba(vocab_size, d_model, n_layers=12)),
        ("Hybrid Mamba", HybridMamba(vocab_size, d_model, n_layers=12))
    ]
    
    for name, model in models:
        train_and_generate(
            model, 
            name, 
            epochs=EPOCHS, 
            device=device, 
            temperature=TEMPERATURE,
            save_dir=save_dir
        )

📱 Dispositivo: cpu
🔤 Vocabulario: 89 caracteres

🚀 ENTRENANDO: Pure Mamba


  Época 1/8:   0%|          | 0/80 [00:00<?, ?it/s]

KeyboardInterrupt: 

---

## 10. 📝 Reflexión

### Pregunta 1: ¿Ganó Mamba?
Observa la pérdida final de cada modelo. ¿Era lo esperado?

### Pregunta 2: Texto generado
Lee el texto generado por cada uno:
- ¿Cuál suena más "a Don Quijote"?
- ¿Hay fragmentos reconocibles?

### Pregunta 3: Eficiencia
Mamba escala O(n), Transformer O(n²). Para secuencias largas:
- ¿Cuándo importa realmente?
- ¿Es esta demo suficientemente larga para verlo?

### Pregunta 4: Selección
La "magia" de Mamba está en que B, C, dt dependen del input.
- ¿Por qué esto es mejor que valores fijos?
- ¿Puedes pensar en ejemplos donde "olvidar" selectivamente es útil?

---

## 11. 🔬 Experimentos opcionales

Una vez que entiendas el código base, prueba:

### A. Cambia el estado
```python
# Estado más grande → más capacidad pero más lento
model = MiniMamba(vocab_size, d_model=128, n_layers=12, d_state=32)
```

### B. Más o menos atención en híbrido
```python
# Cada 2 capas en vez de cada 3
for i in range(n_layers):
    if i % 2 == 0:  # Cambiado de % 3 a % 2
        layers.append(AttentionBlock(...))
```

### C. Temperature
```python
TEMPERATURE = 0.5  # Más seguro
# vs
TEMPERATURE = 1.2  # Más loco
```

### D. ¿Qué pasa si...
```python
# Haces d_state=1?
# Haces d_conv=1?
# Quitas el gate z?
```

---

## 📚 Resumen de lo que has aprendido

✅ **State Space Models (SSM)**: sistemas con memoria de tamaño fijo

✅ **Selección de contenido**: Mamba decide dinámicamente qué recordar

✅ **Complejidad lineal**: Mamba escala O(n), no O(n²)

✅ **Puertas**: z gate y D skip connection

✅ **Trade-offs**: eficiencia vs expresividad

---

> 💬 **Frase para recordar**:
> 
> *"Mamba no es 'sin atención'. Es atención lineal: selecciona qué recordar, cómo recordarlo, y cuándo olvidarlo."*

---

🎉 **¡Completado!**